# 01a · Bad-channel decision and bad-epoch QC

F15 作为重复高方差坏道在重参考前排除；ERP 伪迹指标用于同步清理 ERP/HG trial。

In [ ]:
# [Setup]
from pathlib import Path
import sys, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT=Path('/home/lirui/liulab_project/ieeg/Project_colorieeg_2026'); PIPE=ROOT/'color_cognition_pipeline'/'analyse_0720'
sys.path.insert(0,str(PIPE)); import config
from utils.epochs import load_epochs, save_epochs
from utils.qc import detect_bad_epochs
config.ensure_output_dirs()

In [ ]:
# [Epoch QC] Robust multichannel metrics; save masks, clean data, tables and figures
summary=[]; qc_dir=config.subject_result_dir('test001')/'preprocessing'
for task in config.RUNS:
    ep=load_epochs(config.INTERMEDIATE_ROOT/'test001'/'preprocessing'/f'task{task}_erp.npz')
    keep,metrics=detect_bad_epochs(ep['data'],config.BAD_EPOCH_ROBUST_Z,config.BAD_EPOCH_CHANNEL_FRACTION)
    np.savez_compressed(qc_dir/f'task{task}_epoch_qc.npz',keep=keep,trigger=ep['triggers'],**metrics)
    table=pd.DataFrame({'epoch_index_0based':np.arange(len(keep)),'trigger':ep['triggers'],'keep':keep,**metrics})
    table.to_csv(qc_dir/f'task{task}_epoch_qc.csv',index=False)
    save_epochs(config.INTERMEDIATE_ROOT/'test001'/'preprocessing'/f'task{task}_erp_clean.npz',ep['data'][keep],ep['times_ms'],ep['triggers'][keep],ep['channel_names'],{'task':task,'bad_channels':config.BAD_CHANNELS['test001'],'bad_epoch_z':config.BAD_EPOCH_ROBUST_Z,'rejected_epochs':int((~keep).sum())})
    fig,ax=plt.subplots(figsize=(10,3.8)); ax.plot(metrics['bad_channel_fraction'],lw=.8); ax.scatter(np.flatnonzero(~keep),metrics['bad_channel_fraction'][~keep],color='crimson',s=18,label='Rejected'); ax.axhline(config.BAD_EPOCH_CHANNEL_FRACTION,color='k',ls='--',lw=1); ax.set(xlabel='Epoch index',ylabel='Bad-channel fraction',title=f'Task {task} epoch QC'); ax.legend(frameon=False); fig.tight_layout(); fig.savefig(qc_dir/f'task{task}_epoch_qc.png',dpi=220); plt.show()
    summary.append({'task':task,'epochs_before':len(keep),'epochs_rejected':int((~keep).sum()),'epochs_kept':int(keep.sum()),'rejected_percent':100*(~keep).mean()})
summary=pd.DataFrame(summary); summary.to_csv(qc_dir/'epoch_qc_summary.csv',index=False); display(summary)